In [32]:
import pandas as pd
import numpy as np
import random
import folium
import time
import requests
from IPython.display import display, HTML
import json

# 1. Konfigurasi File Jarak dan Waktu
files_matriks_jarak = {
    'Selatan': '../data/matriks_jarak_riil_selatan.csv',
    'Timur': '../data/matriks_jarak_riil_timur.csv',
    'Utara': '../data/matriks_jarak_riil_utara.csv',
    'Barat': '../data/matriks_jarak_riil_barat.csv',
    'Pusat': '../data/matriks_jarak_riil_pusat.csv'
}

files_matriks_waktu = {
    'Selatan': '../data/datamatriks_waktu_selatan.csv',
    'Timur': '../data/datamatriks_waktu_timur.csv',
    'Utara': '../data/datamatriks_waktu_utara.csv',
    'Barat': '../data/datamatriks_waktu_barat.csv',
    'Pusat': '../data/datamatriks_waktu_pusat.csv'
}

# Baca data koordinat
coords_df = pd.read_csv('../data/koordinat_eas.csv')

def fix_coord(val, is_lat=True):
    val = str(val).strip()
    if not val or val.lower() == 'nan':
        return 0.0
    
    # Hapus semua titik dan koma bawaan Excel
    val = val.replace('.', '').replace(',', '')
    
    try:
        num = float(val)
        if is_lat:
            # Latitude Surabaya harusnya di kisaran -7.xxx
            if num > 0: 
                num = -num # Paksa jadi negatif kalau dataset lupa kasih minus
            while num <= -10:
                num /= 10.0
        else:
            # Longitude Surabaya harusnya di kisaran 112.xxx
            while num >= 1000:
                num /= 10.0
            # Antisipasi kalau ada yang kurang dari 112 (untuk jaga-jaga)
            while num > 0 and num < 112:
                num *= 10.0
        return num
    except:
        return 0.0

# Terapkan fungsi pembersih ke kolom Latitude dan Longitude
coords_df['Latitude'] = coords_df['Latitude'].apply(lambda x: fix_coord(x, is_lat=True))
coords_df['Longitude'] = coords_df['Longitude'].apply(lambda x: fix_coord(x, is_lat=False))

# ====================================================================

# Cek dan tambahkan UPTD Gudang Farmasi
if not coords_df['Nama Puskesmas'].str.contains('Gudang Farmasi', case=False, na=False).any():
    new_row = pd.DataFrame([{
        'Nama Puskesmas': 'UPTD Gudang Farmasi Surabaya', 
        'Latitude': -7.255, 
        'Longitude': 112.750, 
        'Wilayah': 'Pusat'
    }])
    coords_df = pd.concat([coords_df, new_row], ignore_index=True)

# ====================================================================
# FUNGSI PRIORITAS: Bangun Gamma Array dari koordinat_eas.csv
# Skala prioritas:
#   1) Jaringan Pelayanan = Puskesmas (Induk), Jenis Layanan = Rawat Inap  → gamma 3.0
#   2) Jaringan Pelayanan = Puskesmas (Induk), Jenis Layanan = Rawat Jalan → gamma 2.0
#   3) Jaringan Pelayanan = Pustu,             Jenis Layanan = Rawat Jalan → gamma 1.0
# ====================================================================
def build_priority_scores(places, coords_df):
    """
    Lookup setiap nama node ke koordinat_eas.csv dan assign gamma
    berdasarkan kombinasi kolom 'Jaringan Pelayanan' + 'Jenis Layanan'.
    Node depot (Gudang Farmasi) dan yang tidak ditemukan tetap 1.0.
    """
    scores = np.ones(len(places))

    # Buat lookup table sekali (case-insensitive, strip whitespace)
    lookup = coords_df.copy()
    lookup['_key'] = lookup['Nama Puskesmas'].str.strip().str.lower()

    for i, name in enumerate(places):
        name_clean = name.strip().lower()

        # Depot tidak perlu prioritas kunjungan
        if 'gudang farmasi' in name_clean:
            scores[i] = 1.0
            continue

        match = lookup[lookup['_key'] == name_clean]
        if not match.empty:
            jaringan = str(match.iloc[0]['Jaringan Pelayanan']).strip().lower()
            layanan  = str(match.iloc[0]['Jenis Layanan']).strip().lower()

            if 'puskesmas' in jaringan and 'rawat inap' in layanan:
                scores[i] = 3.0   # Prioritas 1 — Puskesmas Induk Rawat Inap
            elif 'puskesmas' in jaringan and 'rawat jalan' in layanan:
                scores[i] = 2.0   # Prioritas 2 — Puskesmas Induk Rawat Jalan
            elif 'pustu' in jaringan:
                scores[i] = 1.0   # Prioritas 3 — Pustu Rawat Jalan
            # else: tetap 1.0 untuk tipe lain yang tidak terdefinisi

    return scores

# ====================================================================

# Fungsi untuk memanggil rute OSRM (Jalan Riil)
def get_route_geometry(start_coords, end_coords):
    url = f"http://router.project-osrm.org/route/v1/driving/{start_coords[1]},{start_coords[0]};{end_coords[1]},{end_coords[0]}?overview=full&geometries=geojson"
    try:
        r = requests.get(url, timeout=5)
        res = r.json()
        if res['code'] == 'Ok':
            geometry = res['routes'][0]['geometry']['coordinates']
            return [[coord[1], coord[0]] for coord in geometry]
    except Exception as e:
        pass
    return [start_coords, end_coords]

In [33]:
def solve_aco(distance_matrix,
              time_matrix,
              node_names=None,
              priority_scores=None,  # <-- TAMBAHAN: array gamma dari koordinat_eas.csv
              num_ants=50,
              max_iter=150,
              alpha=1.0,
              beta=2.0,
              rho=0.1,
              Q=100):
    
    n = len(time_matrix)
    pheromone = np.ones((n, n)) * 0.1
    visibility = np.zeros((n, n))

    # Pre-kalkulasi visibilitas menggunakan MATRIKS WAKTU (1 / waktu)
    for i in range(n):
        for j in range(n):
            if i != j and time_matrix[i][j] > 0:
                visibility[i][j] = 1.0 / time_matrix[i][j]

    # ================================================================
    # Inisialisasi Bobot Prioritas Gamma (γ) Berdasarkan Jenis Puskesmas
    # Didahulukan: priority_scores dari koordinat_eas.csv (akurat).
    # Fallback: keyword matching di node_names (lama, kurang akurat).
    # ================================================================
    gamma_array = np.ones(n)
    if priority_scores is not None:
        # Gunakan skor prioritas yang sudah dihitung dari koordinat_eas.csv
        gamma_array = np.array(priority_scores, dtype=float)
    elif node_names is not None:
        # Fallback keyword matching (dipertahankan untuk keamanan)
        for i, name in enumerate(node_names):
            name_lower = name.lower()
            if "inap" in name_lower:
                gamma_array[i] = 3.0
            elif "jalan" in name_lower or "induk" in name_lower:
                gamma_array[i] = 2.0
            elif "pustu" in name_lower or "pembantu" in name_lower:
                gamma_array[i] = 1.0
    # ================================================================

    best_route = None
    best_time_total = float('inf')
    best_dist_total = float('inf')
    convergence_history = [] 

    for it in range(max_iter):
        routes = []
        route_times = [] # Catat waktu untuk update feromon
        
        current_best_iter_time = float('inf') 

        for ant in range(num_ants):
            route = [0] 
            visited = set([0])
            waktu_kurir = 0 

            while len(visited) < n:
                curr = route[-1]
                probs = np.zeros(n)

                for j in range(n):
                    if j not in visited:
                        # Pengecekan limit 600 menit menggunakan MATRIKS WAKTU
                        waktu_perjalanan = time_matrix[curr][j]
                        waktu_kembali = time_matrix[j][0]
                        
                        if waktu_kurir + waktu_perjalanan + 20 + waktu_kembali <= 600:
                            probs[j] = (pheromone[curr][j] ** alpha) * (visibility[curr][j] ** beta) * gamma_array[j]
                        else:
                            probs[j] = 0.0

                if probs.sum() == 0:
                    unvisited = list(set(range(n)) - visited)
                    if not unvisited: 
                        break
                    
                    if curr == 0:
                        next_node = random.choice(unvisited)
                        waktu_kurir += time_matrix[curr][next_node] + 20
                        route.append(next_node)
                        visited.add(next_node)
                    else:
                        route.append(0)
                        waktu_kurir = 0 
                else:
                    probs = probs / probs.sum()
                    next_node = np.random.choice(range(n), p=probs)

                    # Update akumulasi waktu dengan matriks waktu
                    waktu_kurir += time_matrix[curr][next_node] + 20
                    route.append(next_node)
                    visited.add(next_node)

            if route[-1] != 0:
                route.append(0)

            # Kalkulasi total waktu untuk menilai efisiensi rute ini (Feromon Update)
            total_time = 0
            total_dist = 0
            for i in range(len(route) - 1):
                total_time += time_matrix[route[i]][route[i+1]]
                total_dist += distance_matrix[route[i]][route[i+1]]

            routes.append(route)
            route_times.append(total_time)

            # Simpan rute berdasarkan total waktu paling efisien
            if total_time < best_time_total:
                best_time_total = total_time
                best_dist_total = total_dist
                best_route = route 

            if total_time < current_best_iter_time:
                current_best_iter_time = total_time

        convergence_history.append(best_time_total)

        # Update Feromon berdasarkan Kualitas WAKTU (time_matrix)
        pheromone *= (1 - rho)
        for i in range(num_ants):
            if i < len(routes): 
                d_tau = Q / route_times[i] # Rute waktu paling singkat dapat feromon terbesar
                r = routes[i]
                for j in range(len(r) - 1):
                    pheromone[r[j]][r[j+1]] += d_tau
                    pheromone[r[j+1]][r[j]] += d_tau 

    return best_route, best_dist_total, best_time_total, convergence_history


In [35]:
# 3. Eksekusi Algoritma & Export JSON + Output Teks
results_dist = {}
results_time = {}
results_routes = {}
results_runtime = {}
results_convergence = {}

semua_hasil_json = {
    "algoritma": "ACO",
    "hasil_per_klaster": {}
}

os.makedirs('../output_json', exist_ok=True)

def singkat_nama(nama):
    """Singkat 'Puskesmas X' dan 'Pustu X' jadi 'P. X', depot tetap."""
    nama = nama.strip()
    if nama.lower().startswith('puskesmas '):
        return 'P. ' + nama[10:]
    elif nama.lower().startswith('pustu '):
        return 'P. ' + nama[6:]
    return nama

print("🚀 MEMULAI PROSES ACO VRP...\n")

for region in files_matriks_jarak.keys():
    region_start_time = time.time()

    df_jarak = pd.read_csv(files_matriks_jarak[region], index_col=0)
    df_waktu = pd.read_csv(files_matriks_waktu[region], index_col=0)

    places = df_jarak.index.tolist()
    matrix_jarak = df_jarak.values
    matrix_waktu = df_waktu.values

    np.random.seed(42)
    random.seed(42)

    prio_scores = build_priority_scores(places, coords_df)

    best_route_idx, best_dist, best_time, convergence_history = solve_aco(
                                          distance_matrix=matrix_jarak,
                                          time_matrix=matrix_waktu,
                                          node_names=places,
                                          priority_scores=prio_scores,
                                          num_ants=50,
                                          max_iter=150,
                                          alpha=1.0,
                                          beta=2.0,
                                          rho=0.1,
                                          Q=100)

    region_end_time = time.time()

    # ---------------------------------------------------------
    # PROSES PEMBUATAN STRUKTUR JSON
    # ---------------------------------------------------------
    rute_per_kurir_list = []
    current_route_idx = []
    id_kurir = 1

    for node in best_route_idx:
        current_route_idx.append(node)
        if node == 0 and len(current_route_idx) > 1:
            jarak_kurir = 0
            waktu_kurir = 0
            urutan = []
            koords = []

            for i in range(len(current_route_idx) - 1):
                u = current_route_idx[i]
                v = current_route_idx[i+1]
                jarak_kurir += matrix_jarak[u][v]
                waktu_kurir += matrix_waktu[u][v]
                if v != 0:
                    waktu_kurir += 20

            for idx_puskesmas in current_route_idx:
                nama_tempat = places[idx_puskesmas]
                urutan.append(nama_tempat)

                match = coords_df[coords_df['Nama Puskesmas'].str.strip().str.lower() == nama_tempat.strip().lower()]
                if not match.empty:
                    koords.append([match.iloc[0]['Latitude'], match.iloc[0]['Longitude']])
                else:
                    koords.append([0.0, 0.0])

            rute_per_kurir_list.append({
                "id_kurir": id_kurir,
                "waktu_tempuh_menit": round(waktu_kurir, 2),
                "jarak_tempuh_km": round(jarak_kurir, 2),
                "urutan_kunjungan": urutan,
                "koordinat_kunjungan": koords
            })

            id_kurir += 1
            current_route_idx = [0]

    semua_hasil_json["hasil_per_klaster"][region] = {
        "total_kurir": len(rute_per_kurir_list),
        "waktu_komputasi_detik": round(region_end_time - region_start_time, 3),
        "total_waktu_semua_menit": round(best_time, 2),
        "total_jarak_semua_km": round(best_dist, 2),
        "riwayat_konvergensi": [round(v, 3) for v in convergence_history],
        "rute_per_kurir": rute_per_kurir_list
    }

    results_dist[region] = best_dist
    results_time[region] = best_time
    results_runtime[region] = region_end_time - region_start_time
    results_convergence[region] = convergence_history

    # ---------------------------------------------------------
    # OUTPUT TEKS FORMAT
    # ---------------------------------------------------------
    waktu_komputasi = round(region_end_time - region_start_time, 3)
    jam_total = round(best_time / 60, 2)

    print("=" * 70)
    print(f"📍 KLASTER {region.upper()}")
    print("-" * 70)
    print(f"Total Kurir        : {len(rute_per_kurir_list)} Orang")
    print(f"Waktu Komputasi    : {waktu_komputasi} Detik")
    print(f"Total Jarak Klaster: {round(best_dist, 2)} KM")
    print(f"Total Waktu Klaster: {round(best_time, 2)} Menit (Setara {jam_total} Jam)")
    print("-" * 70)

    for kurir in rute_per_kurir_list:
        rute_singkat = ' ➔ '.join([singkat_nama(n) for n in kurir['urutan_kunjungan']])
        print(f"🚚 [KURIR {kurir['id_kurir']}] - Jarak: {kurir['jarak_tempuh_km']} KM | Waktu: {kurir['waktu_tempuh_menit']} Menit")
        print(f"   Rute: {rute_singkat}\n")

with open('../output_json/rute_aco.json', 'w') as f:
    json.dump(semua_hasil_json, f, indent=4)

print("\nBerhasil! File rute_aco.json sudah terbuat di folder ../output_json")

🚀 MEMULAI PROSES ACO VRP...

📍 KLASTER SELATAN
----------------------------------------------------------------------
Total Kurir        : 2 Orang
Waktu Komputasi    : 17.744 Detik
Total Jarak Klaster: 120.12 KM
Total Waktu Klaster: 187.8 Menit (Setara 3.13 Jam)
----------------------------------------------------------------------
🚚 [KURIR 1] - Jarak: 75.2 KM | Waktu: 586.18 Menit
   Rute: UPTD Gudang Farmasi ➔ P. Sidosermo ➔ P. Jagir ➔ P. Bendul Merisi ➔ P. Margorejo ➔ P. Wonokromo ➔ P. Ketintang ➔ P. Jambangan ➔ P. Karah ➔ P. Gunungsari ➔ P. Kedurus ➔ P. Kebraon ➔ P. Wiyung ➔ P. Babatan ➔ P. Pradah Kali Kendal ➔ P. Sawunggaling ➔ P. Pakis ➔ P. Putat Jaya ➔ P. Dukuh Kupang ➔ P. Putat Jaya ➔ P. Simokatrungan ➔ P. Banyu Urip ➔ P. Petemon ➔ P. Sawahan ➔ UPTD Gudang Farmasi

🚚 [KURIR 2] - Jarak: 44.92 KM | Waktu: 241.62 Menit
   Rute: UPTD Gudang Farmasi ➔ P. Jemursari ➔ P. Siwalankerto ➔ P. Dukuh Menanggal ➔ P. Gayungan ➔ P. Kebonsari ➔ P. Pagesangan ➔ P. Balas Klumprik ➔ P. Warugunung 